In [1]:
import sys, pathlib
import torch
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "rnencodec_notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
from inference.rt import run_inference, play_audio

# 1. Static Examples

# 2. Glissando/slide from one point to another

The idea here will be to look at the different representations and see which can actually do a smoother glissando from one point to another.

### 2.1 Frequency representation

In [2]:
# --- CONFIGURATION ---
MODEL_DIR = "output_26.03.19.18.44"  # Update this to your model path
CHECKPOINT = "last_checkpoint.pt"            # e.g., 'checkpoint_75.pt' or None for latest

print(f"Loading model from: {MODEL_DIR}")

# Create a sequence (10 seconds * 75 fps = 750 frames)
num_frames = int(10 * 75)
# IMPORTANT: num_features must match your conditioning_config.json (e.g., 1 for freq, 11 for class)
num_features = 1

# Create a linear ramp from 0.0 to 1.0 (normalized pitch)
slide_values = torch.linspace(0.0, 1.0, num_frames).view(num_frames, num_features)

print("Rendering pitch slide...")
audio_slide = run_inference(
    model_dir=MODEL_DIR,
    mode="offline",
    conditioning_sequence=slide_values # Use the new argument name!
)
play_audio(audio_slide)

Loading model from: output_26.03.19.18.44
Rendering pitch slide...
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using last checkpoint: checkpoint_75.pt
Loading RNN model from checkpoint_75.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Generating audio in chunks of 8 frames...


### 2.2 Class representation

In [3]:
# --- CONFIGURATION ---
MODEL_DIR = "output_26.03.19.18.44"  # Update this to your model path
CHECKPOINT = "last_checkpoint.pt"            # e.g., 'checkpoint_75.pt' or None for latest

print(f"Loading model from: {MODEL_DIR}")

num_frames = 752  # 10s @ 75fps (approx, multiple of 8)
num_classes = 11  # Based on your model_config.cond_size

# 1. Initialize an empty sequence
slide_values = torch.zeros(num_frames, num_classes)

# 2. Define our "Note Path" (e.g., Note 0 to Note 5)
start_note = 0
end_note = 5

# 3. Create the "Fade" 
# This creates a ramp from 0 to 1 over the 10 seconds
alpha = torch.linspace(0, 1, num_frames)

# At alpha=0: Class 0 is 1.0, Class 5 is 0.0
# At alpha=0.5: Class 0 is 0.5, Class 5 is 0.5 (Maximum Ambiguity)
# At alpha=1: Class 0 is 0.0, Class 5 is 1.0
slide_values[:, start_note] = 1.0 - alpha
slide_values[:, end_note] = alpha

print(f"Rendering probabilistic slide from Note {start_note} to {end_note}...")
audio_slide = run_inference(
    model_dir=MODEL_DIR,
    mode="offline",
    conditioning_sequence=slide_values
)
play_audio(audio_slide)

Loading model from: output_26.03.19.18.44
Rendering probabilistic slide from Note 0 to 5...
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using last checkpoint: checkpoint_75.pt
Loading RNN model from checkpoint_75.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Generating audio in chunks of 8 frames...


AssertionError: cond must be (T,1)

### 2.3 Sine-cosine representation

In [ ]:
# --- CONFIGURATION ---
MODEL_DIR = "output_26.03.19.18.44"  # Update this to your model path
CHECKPOINT = "last_checkpoint.pt"            # e.g., 'checkpoint_75.pt' or None for latest

print(f"Loading model from: {MODEL_DIR}")

num_frames = 752 
# We need 2 features: [Sine, Cosine]
slide_values = torch.zeros(num_frames, 2)

# Create an angle ramp from 0 to Pi (half a circle)
angles = torch.linspace(0, np.pi, num_frames)

slide_values[:, 0] = torch.sin(angles) # Feature 1
slide_values[:, 1] = torch.cos(angles) # Feature 2

print("Rendering Sine-Cosine pitch circle...")
audio_slide = run_inference(
    model_dir=MODEL_DIR,
    mode="offline",
    conditioning_sequence=slide_values # Use the new argument name!
)
play_audio(audio_slide)

### 2.4 Fourier representation

In [ ]:
# --- CONFIGURATION ---
MODEL_DIR = "output_26.03.19.18.44"  # Update this to your model path
CHECKPOINT = "last_checkpoint.pt"            # e.g., 'checkpoint_75.pt' or None for latest

print(f"Loading model from: {MODEL_DIR}")

num_features = 64 # Match your conditioning_size
num_frames = 752

# Let's say we have a Fourier vector for Note A and Note B
# (For this example, we'll just use random anchors)
fourier_start = torch.randn(1, num_features) 
fourier_end = torch.randn(1, num_features)

# Linear interpolation (LERP) between the two signatures
alphas = torch.linspace(0, 1, num_frames).view(-1, 1)
slide_values = (1 - alphas) * fourier_start + alphas * fourier_end

print("Rendering Fourier signature interpolation...")
audio_slide = run_inference(
    model_dir=MODEL_DIR,
    mode="offline",
    conditioning_sequence=slide_values # Use the new argument name!
)
play_audio(audio_slide)

### 2.5 Log-norm representation

In [ ]:
# --- CONFIGURATION ---
MODEL_DIR = "output_26.03.19.18.44"  # Update this to your model path
CHECKPOINT = "last_checkpoint.pt"            # e.g., 'checkpoint_75.pt' or None for latest

print(f"Loading model from: {MODEL_DIR}")

num_frames = 752
num_features = 1
# A linear ramp here sounds like a smooth musical glissando
slide_values = torch.linspace(0.0, 1.0, num_frames).view(num_frames, 1)

print("Rendering Log-Norm musical glissando...")
audio_slide = run_inference(
    model_dir=MODEL_DIR,
    mode="offline",
    conditioning_sequence=slide_values # Use the new argument name!
)
play_audio(audio_slide)